# Biomass flux & production envelopes w/ StandAlone FBA
Last updated: 5/5/26


1) Calculate biomass flux coefficients based on homeostatic dmdt target and estimated values, as in `biomass_flux_comparison.ipynb` (reproducibility sanity check)

2) Use these values to write a new biomass reaction to the stoichiometric matrix

3) Evaluate model in three modes:

    a) Mode A: legacy (homeostatic+kinetic)

    b) Mode B: "WCM-FBA" (drop homeostatic and kinetic, optimize for new biomass flux)

    c) Mode C: "WCM-FBA-kinetic" (reintroduce kinetic objective)
    
    Generate production envelopes for EtOH and Trp and example products

## Setup: Imports and Model Loading

In [3]:
import os
import math
import time as _time
import dill
import numpy as np
import pandas as pd
import requests
import cvxpy as cp
from plotly.graph_objects import Scatter, Figure
from plotly.subplots import make_subplots
from cobra.io import load_json_model

from ecoli.processes.metabolism_redux import NetworkFlowModel, FlowResult
from wholecell.utils import units

# Resolve the vEcoli repo root by walking up from cwd until we find the
# `ecoli/processes/metabolism_redux.py` marker. Lets the notebook run from
# any subdirectory of the vEcoli repo without a user-specific chdir.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, 'ecoli', 'processes', 'metabolism_redux.py')):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        raise RuntimeError('Run this notebook from inside the vEcoli repo')
    _root = _parent
os.chdir(_root)
print(f'cwd: {_root}')

%load_ext autoreload
%autoreload 2

cwd: /Users/chris/projects/SMS/vEcoli
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load model

In [4]:
def load_sim(folder_path):
    """Load timeseries-emitter sim output. Returns fba, bulk, metabolism, output."""
    output = np.load(folder_path + '0_output.npy', allow_pickle=True).item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    with open(folder_path + 'agent_steps.pkl', 'rb') as f:
        agent = dill.load(f)
    metabolism = agent['ecoli-metabolism-redux']
    return fba, bulk, metabolism, output


# Canonical sim — basal metabolism_redux run, t=600
folder_path = 'out/metabolic_engineering/basal/metabolism_redux_metabolic_engineering_600_2026-04-23/'
fba, bulk, metabolism, _ = load_sim(folder_path)

homeostatic_ids = list(metabolism.homeostatic_metabolites)
n_homeo = len(homeostatic_ids)

# Empirical counts ↔ mM conversion factor.
# `target_homeostatic_dmdt_conc / target_homeostatic_dmdt` per (t, met) equals
# `counts_to_molar` (mM/count). The end-of-run snapshot on `metabolism` is biased
# ~7% low vs. the time-mean for this run, so we use the empirical mean as the
# bridge default for output-boundary unit conversions.
target_arr_full      = np.array(fba['target_homeostatic_dmdt'][1:])
target_conc_arr_full = np.array(fba['target_homeostatic_dmdt_conc'][1:])
nonzero = target_arr_full != 0
ratios = np.where(nonzero, target_conc_arr_full / np.where(nonzero, target_arr_full, 1.0), np.nan)
c2m_empirical = float(np.nanmean(ratios))

print(f'sim: {folder_path}')
print(f'  reactions:        {len(metabolism.reaction_names):>6,}')
print(f'  metabolites:      {len(metabolism.metabolite_names):>6,}')
print(f'  homeostatic:      {n_homeo:>6,}')
print(f'  kinetic:          {len(metabolism.kinetic_constraint_reactions):>6,}')
print(f'  timesteps:        {len(fba["target_homeostatic_dmdt"]):>6,}')
print()
print(f'empirical counts_to_molar = {c2m_empirical:.3e} mM/count')
snap = metabolism.counts_to_molar.asNumber()
print(f'  (snapshot for comparison: {snap:.3e}, {100 * (snap / c2m_empirical - 1):+.1f}%)')

sim: out/metabolic_engineering/basal/metabolism_redux_metabolic_engineering_600_2026-04-23/
  reactions:         9,412
  metabolites:       6,149
  homeostatic:         172
  kinetic:             407
  timesteps:           601

empirical counts_to_molar = 1.343e-06 mM/count
  (snapshot for comparison: 1.253e-06, -6.6%)


### Unit conversion helpers — counts ↔ mmol/gDCW

Conversion chain:

```
counts/s × counts_to_molar / rho_dry × 3600 → mmol/(gDCW·h)
                                            ÷ μ → mmol/gDCW (iML1515 unit)
```

Used only at output boundaries. Internal solver math stays in counts/s.

In [5]:
def bridge_constants(metabolism, counts_to_molar=None):
    """Return (rho_dry [gDCW/L], mu [1/s], counts_to_molar [mM/count]).

    Defaults to `c2m_empirical` (time-mean from the listener) — the snapshot
    on the metabolism object is ~7% biased on this run. Pass an explicit
    `counts_to_molar=` to override (e.g. `metabolism.counts_to_molar.asNumber()`
    for snapshot comparison).
    """
    cell_density = metabolism.cell_density.asNumber(units.g / units.L)
    dry_frac     = metabolism.parameters['cell_dry_mass_fraction']
    rho_dry      = cell_density * dry_frac

    doubling_time_s = metabolism.nutrient_to_doubling_time[metabolism.media_id].asNumber(units.s)
    mu = math.log(2) / doubling_time_s

    if counts_to_molar is None:
        counts_to_molar = c2m_empirical

    return rho_dry, mu, counts_to_molar


def counts_per_s_to_mM_per_s(values, metabolism, counts_to_molar=None):
    _, _, c2m = bridge_constants(metabolism, counts_to_molar)
    return values * c2m


def counts_per_s_to_mmol_gdcw_h(values, metabolism, counts_to_molar=None):
    rho_dry, _, c2m = bridge_constants(metabolism, counts_to_molar)
    return values * c2m / rho_dry * 3600


def counts_per_s_to_iml1515_coeff(values, metabolism, counts_to_molar=None):
    """counts/s → mmol/gDCW per unit growth rate (iML1515 biomass-coefficient convention)."""
    rho_dry, mu, c2m = bridge_constants(metabolism, counts_to_molar)
    return values * c2m / (rho_dry * mu)


def v_biomass_user_to_mu_per_h(v_biomass_user, metabolism):
    """v_biomass_user (1.0 = WT) → growth rate μ in 1/h.

    iML1515 reports biomass flux as μ in 1/h. Our LP-internal v_biomass is
    a dimensionless multiplier on WT growth (chosen for solver
    conditioning). This helper converts to the iML1515-comparable unit.
    """
    _, mu_per_s, _ = bridge_constants(metabolism)
    return v_biomass_user * mu_per_s * 3600


# Demo: print the bridge constants and conversion factors using the empirical default.
rho_dry, mu, c2m = bridge_constants(metabolism)
print(f'rho_dry         = {rho_dry:.2f} gDCW/L')
print(f'mu              = {mu:.3e} 1/s   (doubling time = {math.log(2)/mu:.0f} s = {math.log(2)/(mu*3600):.3f} h)')
print(f'mu              = {mu*3600:.3f} 1/h   <-- iML1515 v_biomass convention')
print(f'counts_to_molar = {c2m:.3e} mM/count   (empirical time-mean)')
print(f'\nbridge factors (basal, empirical c2m):')
print(f'  counts/s → mM/s:           x {c2m:.3e}')
print(f'  counts/s → mmol/(gDCW·h):  x {c2m / rho_dry * 3600:.3e}')
print(f'  counts/s → iML1515 coeff:  x {c2m / (rho_dry * mu):.3e}')
print(f'  v_biomass_user → μ (1/h):  x {mu * 3600:.3f}    (1.0 → WT growth rate)')

# For comparison: what the snapshot would have given
c2m_snap = metabolism.counts_to_molar.asNumber()
print(f'\nfor reference, snapshot c2m = {c2m_snap:.3e}   ({100 * (c2m_snap/c2m - 1):+.1f}% vs. empirical)')

rho_dry         = 330.00 gDCW/L
mu              = 2.626e-04 1/s   (doubling time = 2640 s = 0.733 h)
mu              = 0.945 1/h   <-- iML1515 v_biomass convention
counts_to_molar = 1.343e-06 mM/count   (empirical time-mean)

bridge factors (basal, empirical c2m):
  counts/s → mM/s:           x 1.343e-06
  counts/s → mmol/(gDCW·h):  x 1.465e-05
  counts/s → iML1515 coeff:  x 1.549e-05
  v_biomass_user → μ (1/h):  x 0.945    (1.0 → WT growth rate)

for reference, snapshot c2m = 1.253e-06   (-6.6% vs. empirical)


## 1) Calculate biomass flux coefficients based on homeostatic dmdt target and estimated values
as in `biomass_flux_comparison.ipynb` (reproducibility sanity check)

- Extract `target_homeostatic_dmdt` and `estimated_homeostatic_dmdt` from the `metabolism_redux` listener (counts/s, full time-average — matches the prior `biomass_flux_comparison.ipynb` notebook)
- Stability check: targets are roughly static across the cycle for representative mets (water, an amino acid, a cofactor)
- Coefficients-vs-iML1515 scatter: our coefficients vs iML1515 in shared mmol/gDCW units, 60-met intersection, log-log. Confirms order-of-magnitude agreement.
- We compute both `target` and `estimated` as parallel surfaces:
    - `target` is the iML1515-comparison spec (composition target, constraint-free)
    - `estimated` is the envelope-hierarchy default (FBA-achieved demand, contains the WT operating point by construction)
- We take `estimated` (→ `model_e`) forward for the envelope work. Reported behavior on `target` (→ `model_t`) is qualitatively similar; results below use `model_e`.

> **Methods caveat (pocket).** Two metabolites — glycogen-monomer and CPD-12819 (PE lipid) — show a sharp transition around t≈420 in `target_homeostatic_dmdt` from a polymer-catchup process. This inflates their coefficients in `model_t` but not `model_e`, since FBA never achieved those pre-transition targets.

In [6]:
# Time-average homeostatic dmdt vectors over the full cell cycle (drop t=0 row).
# `target` is the iML1515-comparable composition spec; `estimated` is FBA-achieved
# demand. Both kept; estimated taken forward for the envelope work below.
EXTRACTION_START = 1

target_counts = pd.Series(
    np.array(fba['target_homeostatic_dmdt'][EXTRACTION_START:]).mean(axis=0),
    index=homeostatic_ids, name='target_counts_per_s',
)
estimated_counts = pd.Series(
    np.array(fba['estimated_homeostatic_dmdt'][EXTRACTION_START:]).mean(axis=0),
    index=homeostatic_ids, name='estimated_counts_per_s',
)
target_conc = pd.Series(
    np.array(fba['target_homeostatic_dmdt_conc'][EXTRACTION_START:]).mean(axis=0),
    index=homeostatic_ids, name='target_mM_per_s',
)
estimated_conc = pd.Series(
    np.array(fba['estimated_homeostatic_dmdt_conc'][EXTRACTION_START:]).mean(axis=0),
    index=homeostatic_ids, name='estimated_mM_per_s',
)

n_nz_t = int((target_counts.abs() > 0).sum())
n_nz_e = int((estimated_counts.abs() > 0).sum())
print(f'window: t[{EXTRACTION_START}:]  ({len(fba["target_homeostatic_dmdt"]) - EXTRACTION_START} timesteps)')
print(f'  target_counts    nonzero: {n_nz_t:>3} of {len(target_counts)}')
print(f'  estimated_counts nonzero: {n_nz_e:>3} of {len(estimated_counts)}')
print()
print('top 10 |target_counts| (counts/s):')
print(target_counts.abs().sort_values(ascending=False).head(10).to_string())

window: t[1:]  (600 timesteps)
  target_counts    nonzero: 172 of 172
  estimated_counts nonzero: 172 of 172

top 10 |target_counts| (counts/s):
WATER[c]               7.172981e+06
glycogen-monomer[c]    2.115295e+06
CPD-12819[c]           1.343826e+06
PUTRESCINE[c]          4.797771e+05
PPI[c]                 3.549251e+05
CPD-8260[c]            3.224894e+05
ATP[c]                 3.117858e+05
PROTON[c]              3.030712e+05
AMP[c]                 2.981028e+05
MG+2[c]                1.377885e+05


### Stability check — target drift across the run

Plot a few representative mets (high-norm, mid-norm, low-norm by time-mean target) to confirm the targets are roughly static across the cycle. If they are, the time-average is interpretable as a single biomass-composition specification.

In [7]:
# Side-by-side drift plot: target (left) and estimated (right) for the same
# representative mets (high/mid/low norm by time-mean |target|). Confirms
# both spec sources are roughly static across the cycle for the bulk of
# mets, so the time-average is interpretable as a single biomass spec.
estimated_arr_full = np.array(fba['estimated_homeostatic_dmdt'][1:])

target_abs_means = target_counts.abs().sort_values(ascending=False)
drift_mets = (
    target_abs_means.head(3).index.tolist() +
    target_abs_means.iloc[len(target_abs_means)//2 - 1:len(target_abs_means)//2 + 2].index.tolist() +
    target_abs_means.tail(3).index.tolist()
)

# Explicit per-met colors so target (col 1) and estimated (col 2) match.
# Plotly's default colorway advances per-trace, not per-legendgroup, so
# without this both panels would draw the same metabolite in different colors.
palette = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
           '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    subplot_titles=('|target_homeostatic_dmdt|', '|estimated_homeostatic_dmdt|'),
    horizontal_spacing=0.05,
)
for i, met in enumerate(drift_mets):
    color = palette[i % len(palette)]
    idx = homeostatic_ids.index(met)
    fig.add_trace(
        Scatter(y=np.abs(target_arr_full[:, idx]), mode='lines',
                name=met, legendgroup=met, line=dict(color=color)),
        row=1, col=1,
    )
    fig.add_trace(
        Scatter(y=np.abs(estimated_arr_full[:, idx]), mode='lines',
                name=met, legendgroup=met, showlegend=False,
                line=dict(color=color)),
        row=1, col=2,
    )
fig.update_yaxes(type='log', title='|dmdt| (counts/s)', row=1, col=1)
fig.update_yaxes(type='log', row=1, col=2)
fig.update_xaxes(title='timestep', row=1, col=1)
fig.update_xaxes(title='timestep', row=1, col=2)
fig.update_layout(
    title='dmdt across run (counts/s) — high/mid/low norm by target',
    height=500, width=1100,
)
fig.show()

TODO: run for 4-6 gens and take the last 1-2

### Coefficients vs iML1515 (redux of `biomass_flux_comparison.ipynb`)

Convert our biomass coefficients to iML1515's units (mmol/gDCW per unit growth rate) via the bridge, scatter against `BIOMASS_Ec_iML1515_core_75p37M`. Mapping uses the cached MetaCyc→EcoCyc alias map cached locally so we only hit the BiGG API once.

Two scatters: **target** (composition spec, iML1515-comparable) and **estimated** (FBA-achieved demand). They're qualitatively similar; we proceed with `estimated` for downstream envelope work.

The known residual outliers are the energy-currency pairs (ATP/H+ ~17×, ADP/Pi ~50000×) — direct readout of iML1515's gross-hydrolysis stoichiometry vs. vEcoli's net anabolic accounting, *not* a modeling discrepancy.

In [8]:
mapping_cache_dir = 'out/biomass_flux_comparison'
mapping_cache = f'{mapping_cache_dir}/iml1515_to_homeostatic_mapping.csv'

if os.path.exists(mapping_cache):
    mapping = pd.read_csv(mapping_cache)
    print(f'loaded cached mapping: {mapping_cache}   ({len(mapping)} rows)')
else:
    print(f'cache not found at {mapping_cache} — building from BiGG (~7s)')
    os.makedirs(mapping_cache_dir, exist_ok=True)

    model = load_json_model('notebooks/metabolic_engineering/iML1515.json')
    rxn = model.reactions.get_by_id('BIOMASS_Ec_iML1515_core_75p37M')

    rows = []
    for met, coeff in rxn.metabolites.items():
        universal_id = met.id.rsplit('_', 1)[0]
        ecocyc_id = None
        try:
            r = requests.get(
                f'http://bigg.ucsd.edu/api/v2/universal/metabolites/{universal_id}',
                timeout=5,
            )
            if r.ok:
                biocyc = r.json().get('database_links', {}).get('BioCyc', [])
                ecocyc_id = biocyc[0]['id'] if biocyc else None
        except Exception:
            pass
        rows.append({'bigg_id': met.id, 'name': met.name,
                     'coefficient': coeff, 'ecocyc_id': ecocyc_id})
        _time.sleep(0.1)
    mapping = pd.DataFrame(rows).sort_values('coefficient')

    # Compartment + alias maps (MetaCyc→EcoCyc alias logic)
    COMPARTMENT_MAP   = {'c': '[c]', 'p': '[p]', 'e': '[p]'}
    METACYC_TO_ECOCYC = {
        'CPD-15815': 'WATER',
        'CPD-16459': 'Pi',
        'CPD-3':     'AMMONIUM',
        'CPD0-1958': 'L-ORNITHINE',
    }

    def to_vecoli_id(row):
        if not row['ecocyc_id']:
            return None
        frame = row['ecocyc_id'].removeprefix('META:')
        frame = METACYC_TO_ECOCYC.get(frame, frame)
        comp_letter = row['bigg_id'].rsplit('_', 1)[-1]
        return f'{frame}{COMPARTMENT_MAP.get(comp_letter, "[c]")}'

    mapping['vecoli_id'] = mapping.apply(to_vecoli_id, axis=1)
    homeostatic_set = set(homeostatic_ids)
    mapping['in_homeostatic'] = mapping['vecoli_id'].isin(homeostatic_set)

    # Hand-curated alias map for harder cross-compartment cases
    fba_to_homeostatic = {
        '10-FORMYL-THF[c]':                None,
        '2-OCTAPRENYL-6-HYDROXYPHENOL[c]': None,
        'AMMONIA[c]':                      None,
        'AMMONIUM[c]':                     None,
        'CPD-12819[p]':                    'CPD-12819[c]',
        'CPD-17086[p]':                    None,
        'CPD-7[c]':                        None,
        'CPD-9649[c]':                     'UNDECAPRENYL-DIPHOSPHATE[c]',
        'CPD0-2278[p]':                    'CPD-12261[p]',
        'CU+2[c]':                         None,
        'FE+3[c]':                         'FE+3[p]',
        'HSO4[c]':                         None,
        'KDO2-LIPID-IVA[p]':               None,
        '2fe2s_c':                         None,
    }

    def resolve(row):
        key = row['vecoli_id'] if pd.notna(row['vecoli_id']) else row['bigg_id']
        if key in homeostatic_set:
            return key
        return fba_to_homeostatic.get(key)

    mapping['effective_homeostatic_id'] = mapping.apply(resolve, axis=1)
    mapping.to_csv(mapping_cache, index=False)
    print(f'wrote cache: {mapping_cache}   ({len(mapping)} rows)')

mapping = mapping[mapping['effective_homeostatic_id'].notna()].copy()
print(f'matched iML1515 ↔ vEcoli pairs: {len(mapping)}')

loaded cached mapping: out/biomass_flux_comparison/iml1515_to_homeostatic_mapping.csv   (70 rows)
matched iML1515 ↔ vEcoli pairs: 60


In [9]:
def coeff_scatter(x, y, x_label, y_label, title, width=720, height=560):
    """Log-log abs-valued scatter of two coefficient series sharing an index."""
    df = pd.DataFrame({'x': x.abs(), 'y': y.abs()}).dropna()
    df = df[(df['x'] > 0) & (df['y'] > 0)]
    lo = min(df['x'].min(), df['y'].min())
    hi = max(df['x'].max(), df['y'].max())
    fig = Figure([
        Scatter(
            x=df['x'], y=df['y'], mode='markers',
            text=df.index.astype(str),
            hovertemplate='%{text}<br>x: %{x:.3g}<br>y: %{y:.3g}<extra></extra>',
        ),
        Scatter(
            x=[lo, hi], y=[lo, hi], mode='lines',
            line=dict(dash='dash', color='gray'), showlegend=False,
        ),
    ])
    fig.update_xaxes(type='log', title=x_label)
    fig.update_yaxes(type='log', title=y_label)
    fig.update_layout(title=title, width=width, height=height)
    return fig, df


# Convert our coefficients to iML1515 units once
target_iml    = counts_per_s_to_iml1515_coeff(target_counts,    metabolism)
estimated_iml = counts_per_s_to_iml1515_coeff(estimated_counts, metabolism)

# 1) Estimated vs target — full homeostatic set, our internal consistency
fig1, _ = coeff_scatter(
    target_iml, estimated_iml,
    '|target coeff| (mmol/gDCW)',
    '|estimated coeff| (mmol/gDCW)',
    f'Our coefficients: estimated vs target ({len(target_iml)} homeostatic mets)',
)

# 2 & 3) vs iML1515 — 60-met intersection from the cached mapping
df_map = (
    mapping[['effective_homeostatic_id', 'coefficient']]
    .drop_duplicates(subset='effective_homeostatic_id')
    .dropna()
    .set_index('effective_homeostatic_id')
)
iml_series = df_map['coefficient'].abs()

fig2, _ = coeff_scatter(
    iml_series, target_iml.reindex(iml_series.index),
    '|iML1515 biomass coeff| (mmol/gDCW)',
    '|target coeff| (mmol/gDCW)',
    f'iML1515 vs target ({len(iml_series)}-met intersection)',
)
fig3, _ = coeff_scatter(
    iml_series, estimated_iml.reindex(iml_series.index),
    '|iML1515 biomass coeff| (mmol/gDCW)',
    '|estimated coeff| (mmol/gDCW)',
    f'iML1515 vs estimated ({len(iml_series)}-met intersection)',
)

fig1.show()
fig2.show()
fig3.show()

## 2) Use these values to write a new biomass reaction to the stoichiometric matrix


Append one column to `S_orig`: entries are `-coeff_biomass[i]` for each homeostatic metabolite *i* (consumption when `v_biomass > 0`), zeros elsewhere. Append `'BIOMASS'` to the reaction list. Single new flux variable is `v[-1] = v_biomass`.

**We use `estimated_counts` as the biomass-column coefficients** (→ `model_e`). This is the FBA-achieved demand — the existing WT wrapper's actual draw on each homeostatic metabolite, time-averaged across the full cycle. Two reasons:

- **Contains the WT operating point by construction.** Mode B at `v_biomass = 1` reproduces what the wrapper itself was doing.
- **Robust to the polymer-artifact transition** for glycogen-monomer + CPD-12819. FBA never achieved those pre-transition target values, so they don't inflate `estimated` the way they would inflate `target`.

WT normalization: `v_biomass_user = 1` reproduces `estimated_counts` exactly as the metabolite draw rate. In physical units, this corresponds to the WT growth rate **μ ≈ 0.945 1/h** (basal-media doubling time ≈ 2640 s — computed in the bridge cell above), which is the iML1515-comparable convention. The bridge helper `v_biomass_user_to_mu_per_h()` does the explicit conversion.

In [10]:
def augment_with_biomass(S, metabolites, reaction_names, coeff_biomass, homeostatic_ids):
    """Append a biomass column to S. Returns (S_aug, reaction_names_aug, biomass_idx)."""
    n_mets = S.shape[0]
    biomass_col = np.zeros((n_mets, 1))
    homeo_idx = [metabolites.index(m) for m in homeostatic_ids]
    biomass_col[homeo_idx, 0] = -coeff_biomass.values  # consumption
    S_aug = np.concatenate([S, biomass_col], axis=1)
    reaction_names_aug = list(reaction_names) + ['BIOMASS']
    return S_aug, reaction_names_aug, S_aug.shape[1] - 1


S_orig = np.asarray(metabolism.stoichiometry.todense())
metabolites = list(metabolism.metabolite_names)
reaction_names = list(metabolism.reaction_names)

# Scale the biomass column for solver conditioning.
# Raw coefficients range ~9 OOM (2e-2 to 7e6 counts/s for WATER). With
# upper_flux_bound=1e9, that creates dm contributions up to 7e15 in the
# biomass column alone — both GLOP and CLARABEL fail to converge on the
# resulting stretched polytope. Scaling the column by its max coefficient
# brings entries to [~3e-9, 1] (well-conditioned) at the cost of a
# different LP-internal v_biomass interpretation: WT becomes
# v_biomass_internal = biomass_scale instead of 1. We rescale on output
# so user-facing v_biomass_user keeps the WT=1 normalization.
biomass_scale = float(estimated_counts.abs().max())

S_aug, reaction_names_aug, biomass_idx = augment_with_biomass(
    S_orig, metabolites, reaction_names,
    estimated_counts / biomass_scale,
    homeostatic_ids,
)
print(f'S_orig.shape: {S_orig.shape}')
print(f'S_aug.shape:  {S_aug.shape}   (biomass column at idx {biomass_idx})')
print(f'biomass column nonzeros: {(S_aug[:, biomass_idx] != 0).sum()} (should be {len(homeostatic_ids)})')
print(f'biomass column range:    [{np.abs(S_aug[:, biomass_idx][S_aug[:, biomass_idx] != 0]).min():.3e}, '
      f'{np.abs(S_aug[:, biomass_idx]).max():.3e}]')
print(f'\nbiomass_scale = {biomass_scale:.3e} counts/s   (LP-internal WT v_biomass; max coeff = WATER)')
print(f'  v_biomass_internal -> v_biomass_user: divide by biomass_scale')
print(f'  v_biomass_user = 1   <-- WT operating point')

S_orig.shape: (6149, 9412)
S_aug.shape:  (6149, 9413)   (biomass column at idx 9412)
biomass column nonzeros: 172 (should be 172)
biomass column range:    [3.175e-09, 1.000e+00]

biomass_scale = 7.350e+06 counts/s   (LP-internal WT v_biomass; max coeff = WATER)
  v_biomass_internal -> v_biomass_user: divide by biomass_scale
  v_biomass_user = 1   <-- WT operating point


### Round-trip sanity check — what's actually in S_aug

Pull the biomass column straight out of `S_aug[:, biomass_idx]`, undo the column scaling, and scatter the recovered coefficients against iML1515 in mmol/gDCW units. This is the iML1515-vs-estimated scatter from §1, but derived **from the post-augmentation S matrix** rather than from the listener directly. Confirms the column we just built represents the cycle-averaged `estimated_homeostatic_dmdt` we extracted in §1, with no implicit sign or scaling errors introduced by `augment_with_biomass`.

In [11]:
# Recover counts/s coefficients straight from the S_aug biomass column.
# Column entries are -coeff_biomass[i]/biomass_scale for homeostatic mets;
# multiply by -biomass_scale to undo both the sign convention and the
# solver-conditioning scaling.
biomass_col = S_aug[:, biomass_idx]
homeo_idx_arr = np.array([metabolites.index(m) for m in homeostatic_ids])

coeff_from_S_counts = pd.Series(
    -biomass_col[homeo_idx_arr] * biomass_scale,
    index=homeostatic_ids,
    name='from_S_counts_per_s',
)

# Sanity: should match estimated_counts to numerical precision
max_diff = float((coeff_from_S_counts - estimated_counts).abs().max())
print(f'max |round-trip diff| vs estimated_counts: {max_diff:.3e}   (expect ~0)')

# Convert to iML1515 units and scatter against iML1515 on the 60-met intersection
coeff_from_S_iml = counts_per_s_to_iml1515_coeff(coeff_from_S_counts, metabolism)
fig_S, _ = coeff_scatter(
    iml_series, coeff_from_S_iml.reindex(iml_series.index),
    '|iML1515 biomass coeff| (mmol/gDCW)',
    '|biomass column coeff (from S_aug)| (mmol/gDCW)',
    f'iML1515 vs biomass column (round-trip from S_aug, {len(iml_series)}-met intersection)',
)
fig_S.show()

max |round-trip diff| vs estimated_counts: 1.819e-12   (expect ~0)


## 3) Evaluate the augmented model

Three modes for the augmented stoichiometry:

- **Mode A — legacy multi-objective.** Existing wrapper (homeostatic + kinetic + secretion). Biomass column inert. Used as a regression baseline — confirms appending the column doesn't perturb existing flux distributions.
- **Mode B — WCM-FBA.** Drop the homeostatic loss, add a hard `dm[homeostatic] == 0` mass balance constraint, maximize `v_biomass`. No kinetic objective. Structurally equivalent to a classical FBA biomass-max under our WCM-derived coefficients.
- **Mode C — WCM-FBA + soft kinetic.** Mode B plus the kinetic loss term from `metabolism_redux.py:1041-1071` re-added at sweepable weight `w_kin` (piecewise: heavy outside the [lower, upper] band, light inside). Replaces the original hard-upper Mode C, which was structurally over-constrained at biological glucose.

Sections below split this into:
- **§3a** — biomass-max growth rate under each mode (the μ_max story)
- **§3b** — production envelopes (the Pareto-front story; depends on Mode B/C)

### Setup — augmented model, listener arrays, `biomass_max_solve()`

Time-average the kinetic targets, kinetic bounds, and NGAM target from the listener over the full cycle (same window as §1). Construct the augmented `NetworkFlowModel` from `S_aug`. Define `biomass_max_solve()` with optional soft kinetic loss (Mode B = no kinetic loss; Mode C = piecewise loss term at sweepable weight `w_kin`, mirroring `metabolism_redux.py:1041-1071`).

> **Implementation note.** `NetworkFlowModel.__init__` casts the stoichiometry matrix to int64 (`metabolism_redux.py:906`), which silently truncates the fractional biomass-column coefficients to zero. We override `model.S_orig` with the float64 version after construction. The original wrapper's path is unaffected (its stoichiometry is integer-valued).

In [12]:
from scipy.sparse import csr_matrix as _csr

# Time-averaged listener arrays (full cycle, matches §1)
kinetic_reaction_ids = list(metabolism.kinetic_constraint_reactions)
kin_target_full = np.array(fba['target_kinetic_fluxes'][1:]).mean(axis=0)
kin_bounds_full = np.array(fba['target_kinetic_bounds'][1:]).mean(axis=0)
kinetic_targets_arr = np.column_stack([kin_bounds_full[:, 0], kin_target_full, kin_bounds_full[:, 1]])

homeostatic_counts_arr = bulk.iloc[1:, metabolism.homeostatic_metabolite_idx].mean(axis=0).values
homeostatic_concs_arr  = homeostatic_counts_arr * c2m_empirical
homeostatic_dm_targets = estimated_counts.values
maintenance = float(np.mean(fba['maintenance_target'][1:]))

# Build the augmented NetworkFlowModel from S_aug
model_e = NetworkFlowModel(
    stoich_arr=S_aug,
    metabolites=metabolites,
    reactions=reaction_names_aug,
    homeostatic_metabolites=homeostatic_ids,
    kinetic_reactions=kinetic_reaction_ids,
    get_mass=metabolism.parameters['get_mass'],
    gam=metabolism.gam.asNumber(),
)
model_e.set_up_exchanges(
    exchanges=set(metabolism.exchange_molecules) | set(metabolism.allowed_exchange_uptake),
    uptakes=set(metabolism.allowed_exchange_uptake),
)
# Patch: NetworkFlowModel.__init__ casts stoich_arr to int64 (metabolism_redux.py:906),
# silently truncating the fractional biomass-column coefficients to zero. Override
# with the float64 version after construction.
model_e.S_orig = _csr(S_aug.astype(np.float64))

# Glucose exchange index + iML1515-conventional bound (10 mmol/gDCW/h)
exchange_metabolites = list(model_e.exchanges)
glucose_exch_idx = exchange_metabolites.index('GLC[p] exchange')
glucose_bound_mM_s     = 10.0 * rho_dry / 3600.0
glucose_bound_counts_s = glucose_bound_mM_s / c2m_empirical


def biomass_max_solve(
    model, biomass_idx, biomass_scale, ngam_target,
    *,
    mode='B',
    w_kin=0.0, w_kin_in_range=1e-2,
    kinetic_targets=None,
    glucose_bound_counts_s=None, glucose_exch_idx=None,
    fix_v_biomass_user=None,
    secrete_exchange=None,
    efficiency_weight=1e-9,
    upper_flux_bound=1e9,
    solver=cp.HIGHS,
):
    """Standalone-FBA biomass-max solve with optional soft kinetic loss.

    Mode B: w_kin=0 → no kinetic objective.
    Mode C: w_kin>0 → soft kinetic loss mirroring metabolism_redux.py:1041-1071.
        Piecewise norm1 with a hard-bounded in-band variable (free zone between
        kinetic [lower, upper]) and an unbounded outside-band variable (heavy
        penalty); in-band weight = w_kin × w_kin_in_range.

    Optional knobs:
      glucose_bound_counts_s, glucose_exch_idx — cap glucose uptake (counts/s)
      fix_v_biomass_user                       — pin v_biomass for envelope sweeps
      secrete_exchange                         — int index into model.exchanges:
                                                 max product instead of biomass
      efficiency_weight                        — pFBA-style L1 on |v|; small
                                                 weight (1e-9) breaks futile-cycle
                                                 degeneracy and stabilizes HiGHS
    """
    v  = cp.Variable(model.n_orig_rxns)
    e  = cp.Variable(model.n_exch_rxns)
    dm = model.S_orig @ v + model.S_exch @ e
    total_maintenance = ngam_target + model.gam * e @ model.exchange_masses

    constr = [
        dm[model.intermediates_idx] == 0,
        dm[model.homeostatic_idx] == 0,
        v >= 0, v <= upper_flux_bound,
        e >= 0, e <= upper_flux_bound,
    ]
    if model.maintenance_idx is not None:
        constr.append(v[model.maintenance_idx] == total_maintenance)
        constr.append(v[model.maintenance_idx] >= ngam_target)
    if glucose_bound_counts_s is not None and glucose_exch_idx is not None:
        constr.append(e[glucose_exch_idx] <= glucose_bound_counts_s)
    if fix_v_biomass_user is not None:
        constr.append(v[biomass_idx] == fix_v_biomass_user * biomass_scale)

    soft_kin = None
    if mode == 'C' and w_kin > 0 and kinetic_targets is not None:
        kin_idx = model.kinetic_rxn_idx
        target_central = kinetic_targets[:, 1]
        nonzero_target = target_central.copy()
        nonzero_target[nonzero_target == 0] = 1.0
        lower_diff = kinetic_targets[:, 0] - target_central
        upper_diff = kinetic_targets[:, 2] - target_central
        n_kin = len(kin_idx)
        v_diff_in  = cp.Variable(n_kin)
        v_diff_out = cp.Variable(n_kin)
        constr.append(v[kin_idx] == target_central + v_diff_in + v_diff_out)
        constr.append(v_diff_in >= lower_diff)
        constr.append(v_diff_in <= upper_diff)
        soft_kin = (v_diff_in, v_diff_out, nonzero_target)

    if secrete_exchange is None:
        loss = -v[biomass_idx] / biomass_scale
    else:
        loss = -e[secrete_exchange]
    if efficiency_weight is not None and efficiency_weight > 0:
        loss = loss + efficiency_weight * (cp.sum(v) - v[biomass_idx])
    if soft_kin is not None:
        v_diff_in, v_diff_out, nonzero_target = soft_kin
        loss = loss + w_kin * (
            cp.norm1(v_diff_out / nonzero_target)
            + w_kin_in_range * cp.norm1(v_diff_in / nonzero_target)
        )

    p = cp.Problem(cp.Minimize(loss), constr)
    p.solve(solver=solver, verbose=False)

    if p.status != 'optimal':
        return {'status': p.status, 'v_biomass_user': None,
                'mu_per_h': None, 'product_counts_s': None,
                'velocities': None, 'exchanges': None, 'objective': None}

    velocities = np.array(v.value)
    exchanges  = np.array(e.value)
    v_biomass_user = float(velocities[biomass_idx]) / biomass_scale
    return {
        'status': p.status,
        'v_biomass_user': v_biomass_user,
        'mu_per_h': v_biomass_user_to_mu_per_h(v_biomass_user, metabolism),
        'product_counts_s': (float(exchanges[secrete_exchange])
                             if secrete_exchange is not None else None),
        'velocities': velocities,
        'exchanges': exchanges,
        'objective': float(p.value),
    }


print(f'kinetic_targets_arr.shape:     {kinetic_targets_arr.shape}')
print(f'maintenance:                   {maintenance:.4g}')
print(f'glucose exchange:              {exchange_metabolites[glucose_exch_idx]!r}')
print(f'  cap @ 10 mmol/gDCW/h →       {glucose_bound_counts_s:.3e} counts/s')
print(f'model_e ready for Mode A/B/C solves.')

kinetic_targets_arr.shape:     (407, 3)
maintenance:                   3.697e+06
glucose exchange:              'GLC[p] exchange'
  cap @ 10 mmol/gDCW/h →       6.828e+05 counts/s
model_e ready for Mode A/B/C solves.


### §3a — Biomass-max growth rate under each mode

### Mode A — regression check

Run the existing wrapper's multi-objective solve (homeostatic + kinetic + secretion) on the *augmented* model, with the biomass column inert. Confirms appending the column doesn't perturb the existing flux distribution. Expected: `v_biomass ≈ 0` (nothing in the objective rewards biomass flux), top-10 fluxes match `Standalone_FBA.ipynb`.

In [13]:
# Mode A regression — biomass column inert under the existing wrapper objective
# (homeostatic + kinetic + secretion). v_biomass should sit ~0 since nothing
# in the objective rewards biomass flux; flux distribution should match
# Standalone_FBA.ipynb's solve.
objective_weights_modeA = {'secretion': 1e-3, 'kinetics': 1e-7, 'kinetics_in_range': 1e-2}

solution_A = model_e.solve(
    homeostatic_concs=homeostatic_concs_arr,
    homeostatic_dm_targets=homeostatic_dm_targets,
    ngam_target=maintenance,
    kinetic_targets=kinetic_targets_arr,
    binary_kinetic_idx=None,
    objective_weights=objective_weights_modeA,
    upper_flux_bound=1_000_000_000,
    solver=cp.GLOP,
)
v_A = np.array(solution_A.velocities)
v_biomass_A_user = v_A[biomass_idx] / biomass_scale

print(f'Mode A — augmented model under existing wrapper objective')
print(f'  status:                       optimal')
print(f'  objective:                    {solution_A.objective:.4g}')
print(f'  v_biomass_user:               {v_biomass_A_user:.4g}   (expect ~0; column is inert in Mode A)')
print(f'  predicted μ (1/h):            {v_biomass_user_to_mu_per_h(v_biomass_A_user, metabolism):.4g}')
print(f'  nonzero fluxes:               {int((np.abs(v_A) > 0).sum()):,} of {len(v_A):,}')
print(f'  top-10 |flux| (counts/s):')
top10 = pd.Series(np.abs(v_A), index=reaction_names_aug).sort_values(ascending=False).head(10)
for name, val in top10.items():
    print(f'    {val:>12,.4g}   {name}')

Mode A — augmented model under existing wrapper objective
  status:                       optimal
  objective:                    55.5
  v_biomass_user:               0   (expect ~0; column is inert in Mode A)
  predicted μ (1/h):            0
  nonzero fluxes:               745 of 9,413
  top-10 |flux| (counts/s):
       4.381e+06   maintenance_reaction
       3.803e+06   ATPSYN-RXN (reverse)
       1.728e+06   NADH-DEHYDROG-A-RXN-NADH/UBIQUINONE-8/PROTON//NAD/CPD-9956/PROTON.46.
       1.039e+06   TRANS-RXN0-474
       1.039e+06   RXN0-5268-CPD-9956/OXYGEN-MOLECULE/PROTON//UBIQUINONE-8/WATER/PROTON.59.
       9.923e+05   TRANS-RXN0-545[CCO-PM-BAC-NEG]-CARBON-DIOXIDE//CARBON-DIOXIDE.47. (reverse)
       6.848e+05   GAPOXNPHOSPHN-RXN
       6.848e+05   PHOSGLYPHOS-RXN (reverse)
       6.515e+05   RXN-15511 (reverse)
       6.515e+05   RXN-15510 (reverse)


### Mode B — μ_max vs glucose uptake bound

Drop the kinetic objective entirely (`w_kin = 0`); maximize `v_biomass` at varying glucose uptake bounds. Establishes the carbon-balance scaling. Expected: linear in GUR up to ~15 mmol/gDCW/h, plateaus above. At iML1515-conventional GUR=10, μ ≈ 1.10 1/h (~16% headroom above WT).

In [14]:
# Mode B — μ_max vs glucose uptake bound
# Drop kinetic loss entirely (w_kin=0). Sweep glucose multiplier; biomass-max.
glucose_multipliers = [0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 2.0, 3.0]
mu_vs_gur = []
for m in glucose_multipliers:
    r = biomass_max_solve(
        model_e, biomass_idx, biomass_scale, maintenance,
        mode='B',
        glucose_bound_counts_s=m * glucose_bound_counts_s,
        glucose_exch_idx=glucose_exch_idx,
    )
    gur_mmol_gdcw_h = m * 10.0  # multiplier × iML1515 default GUR
    mu = r['mu_per_h'] if r['status'] == 'optimal' else None
    mu_vs_gur.append({'GUR_mmol_gdcw_h': gur_mmol_gdcw_h, 'mu': mu, 'status': r['status']})
    print(f'  GUR={gur_mmol_gdcw_h:>5.1f}  μ={mu}  status={r["status"]}')

df_gur = pd.DataFrame(mu_vs_gur)
print(); print(df_gur.to_string(index=False))

fig = Figure([
    Scatter(
        x=df_gur['GUR_mmol_gdcw_h'], y=df_gur['mu'],
        mode='lines+markers',
        line=dict(color='#1f77b4', width=2.5),
        marker=dict(size=9),
    ),
])
# Reference lines
fig.add_hline(y=0.945, line_dash='dot', line_color='gray',
              annotation_text='WT μ ≈ 0.945 1/h', annotation_position='right')
fig.add_vline(x=10.0, line_dash='dot', line_color='gray',
              annotation_text='iML1515 GUR=10', annotation_position='top')
fig.update_layout(
    title='Mode B — biomass-max μ vs glucose uptake bound',
    xaxis_title='Glucose uptake bound (mmol/gDCW/h)',
    yaxis_title='μ (1/h)',
    template='plotly_white', width=820, height=480,
)
fig.show()

  GUR=  3.0  μ=0.1226482286877599  status=optimal
  GUR=  5.0  μ=0.3900232683170521  status=optimal
  GUR=  7.0  μ=0.6573983079463422  status=optimal
  GUR= 10.0  μ=1.058460867390275  status=optimal
  GUR= 13.0  μ=1.4595234268342279  status=optimal
  GUR= 15.0  μ=1.726898466463517  status=optimal
  GUR= 20.0  μ=2.3953360655367324  status=optimal
  GUR= 30.0  μ=3.732211263683217  status=optimal

 GUR_mmol_gdcw_h       mu  status
             3.0 0.122648 optimal
             5.0 0.390023 optimal
             7.0 0.657398 optimal
            10.0 1.058461 optimal
            13.0 1.459523 optimal
            15.0 1.726898 optimal
            20.0 2.395336 optimal
            30.0 3.732211 optimal


### Mode C — μ_max vs soft kinetic weight (phase transition)

Re-introduce the kinetic objective as a sweepable soft penalty (`w_kin`). Sweep at fixed GUR=10. Expected: a sharp phase transition between w≈5e-4 and 1e-3, where the LP transitions from "kinetic penalty doesn't bite" to "kinetic penalty collapses biomass."

This locates where the WCM kinetic constraints actually bind under the biomass-max objective.

In [15]:
# Mode C — μ_max vs soft kinetic weight w_kin (phase transition)
# Sweep at fixed GUR=10 mmol/gDCW/h, biomass-max objective with soft kinetic loss.
w_grid = [0.0, 1e-4, 3e-4, 5e-4, 7e-4, 1e-3, 3e-3]

mu_max_kin = []
for w in w_grid:
    r = biomass_max_solve(
        model_e, biomass_idx, biomass_scale, maintenance,
        mode='C', w_kin=w, kinetic_targets=kinetic_targets_arr,
        glucose_bound_counts_s=glucose_bound_counts_s,
        glucose_exch_idx=glucose_exch_idx,
    )
    mu = r['mu_per_h'] if r['status'] == 'optimal' else None
    mu_max_kin.append({'w_kin': w, 'mu_max': mu, 'status': r['status']})
    print(f'  w_kin={w:.0e}  μ_max={mu}  status={r["status"]}')

df_kin = pd.DataFrame(mu_max_kin)
print(); print(df_kin.to_string(index=False))

# Plot — log x for w_kin (treat 0 as a small positive for the log axis)
nonzero_w = [w for w in w_grid if w > 0]
w0_x = min(nonzero_w) / 10.0
fig = Figure([
    Scatter(
        x=df_kin['w_kin'].replace(0, w0_x),
        y=df_kin['mu_max'],
        mode='lines+markers',
        line=dict(color='#2c3e50', width=2.5),
        marker=dict(size=9, color='#2c3e50'),
        showlegend=False,
    ),
])
fig.update_xaxes(
    type='log',
    title='Soft kinetic weight w<sub>kin</sub>',
    exponentformat='power', showexponent='all', dtick=1,
)
fig.update_yaxes(title='μ<sub>max</sub> (1/h)')
fig.update_layout(
    title='Mode C — biomass-max growth rate vs soft kinetic weight (GUR=10)',
    template='plotly_white', width=820, height=480,
    # annotations=[dict(
    #     x=w0_x, y=df_kin['mu_max'].iloc[0],
    #     text='w=0 (Mode B)', showarrow=True, arrowhead=2,
    #     ax=40, ay=-30, font=dict(size=11),
    # )],
)
fig.show()

  w_kin=0e+00  μ_max=1.058460867390275  status=optimal
  w_kin=1e-04  μ_max=1.0503296673692926  status=optimal
  w_kin=3e-04  μ_max=0.9982956338930993  status=optimal
  w_kin=5e-04  μ_max=0.7648595870538787  status=optimal
  w_kin=7e-04  μ_max=0.44771595777909834  status=optimal
  w_kin=1e-03  μ_max=0.04053037352409857  status=optimal
  w_kin=3e-03  μ_max=0.0011064513536186065  status=optimal

 w_kin   mu_max  status
0.0000 1.058461 optimal
0.0001 1.050330 optimal
0.0003 0.998296 optimal
0.0005 0.764860 optimal
0.0007 0.447716 optimal
0.0010 0.040530 optimal
0.0030 0.001106 optimal


### §3b — Production envelopes

The Pareto front of biomass-vs-product secretion at GUR=10. Two products: **ETOH[p]** (canonical fermentation byproduct) and **TRP[p]** (an aromatic amino acid relevant to heterologous biosynthesis pathways).

- **Mode B**: linear envelopes set by carbon balance.
- **Mode C**: family of envelopes parameterized by `w_kin`, with the procedural vertical drop marking each weight's biomass-max corner.
- **Screen**: 24 native secretion exchanges, kinetic-sensitivity test.

### Mode B — linear envelopes (ETOH, TRP)

Sweep `v_biomass` from 0 to `v_max(w=0)` at GUR=10. At each fixed v_biomass, maximize product secretion. Plot ETOH and TRP envelopes side-by-side.

In [16]:
# Mode B — production envelopes for ETOH and TRP at GUR=10.
# For each product, sweep v_biomass from 0 to v_max(w=0); at each fixed v, max product.
PRODUCTS = [
    ('ETOH[p] exchange rev', 'ETOH'),
    ('TRP[p] exchange rev',  'TRP'),
]

def to_mmol(counts_s):
    return counts_s * c2m_empirical * 3600 / rho_dry if counts_s is not None else None

# Mode B v_max (no kinetic loss, biomass-max at GUR=10)
r_modeB = biomass_max_solve(
    model_e, biomass_idx, biomass_scale, maintenance,
    mode='B',
    glucose_bound_counts_s=glucose_bound_counts_s,
    glucose_exch_idx=glucose_exch_idx,
)
v_max_modeB = r_modeB['v_biomass_user']
print(f'Mode B v_biomass_max = {v_max_modeB:.4f}  (μ = {r_modeB["mu_per_h"]:.3f} 1/h)')

n_steps = 15
v_grid = np.linspace(0, v_max_modeB, n_steps)
envelopes_modeB = {}
for exch_id, short in PRODUCTS:
    secrete_idx = exchange_metabolites.index(exch_id)
    products = []
    for v in v_grid:
        r = biomass_max_solve(
            model_e, biomass_idx, biomass_scale, maintenance,
            mode='B',
            glucose_bound_counts_s=glucose_bound_counts_s,
            glucose_exch_idx=glucose_exch_idx,
            secrete_exchange=secrete_idx,
            fix_v_biomass_user=float(v),
        )
        products.append(to_mmol(r['product_counts_s']) if r['status'] == 'optimal' else None)
    envelopes_modeB[short] = (v_grid, products)
    print(f'  {short}: P @ v=0 = {products[0]:.2f},  P @ v_max = {products[-1]:.2f} mmol/gDCW/h')

# Plot side by side
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[s for _, s in PRODUCTS],
                    horizontal_spacing=0.10)
for col_i, (exch_id, short) in enumerate(PRODUCTS, start=1):
    v, p = envelopes_modeB[short]
    fig.add_trace(
        Scatter(x=v * 0.945, y=p, mode='lines+markers',
                line=dict(color='#1f77b4', width=2.5),
                marker=dict(size=8), showlegend=False),
        row=1, col=col_i,
    )
    fig.update_xaxes(title='μ (1/h)', row=1, col=col_i)
    fig.update_yaxes(title=f'{short} secretion (mmol/gDCW/h)', row=1, col=col_i)
fig.update_layout(
    title='Mode B production envelopes (GUR=10)',
    template='plotly_white', height=480, width=1100,
)
fig.show()

Mode B v_biomass_max = 1.1198  (μ = 1.058 1/h)
  ETOH: P @ v=0 = 17.15,  P @ v_max = -0.00 mmol/gDCW/h
  TRP: P @ v=0 = 3.71,  P @ v_max = 0.00 mmol/gDCW/h


In [17]:
import cobra
iml1515 = cobra.io.load_json_model('notebooks/metabolic_engineering/iML1515.json')


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x11298aa20>>
Traceback (most recent call last):
  File "/Users/chris/projects/SMS/vEcoli/.venv/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [18]:
iml1515.medium

{'EX_pi_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_h_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_glc__D_e': 10.0,
 'EX_zn2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_ca2_e': 1000.0,
 'EX_ni2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_sel_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_h2o_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_k_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_cl_e': 1000.0,
 'EX_o2_e': 1000.0,
 'EX_tungs_e': 1000.0,
 'EX_slnt_e': 1000.0}

In [19]:
for target in ['BIOMASS_Ec_iML1515_core_75p37M','EX_etoh_e', 'EX_trp__L_e']:                          
    fba_model = iml1515.copy()                        
    fba_model.objective = target                      
    solution = cobra.flux_analysis.pfba(fba_model)                        
    print(f'{target:40s}  flux = {solution.fluxes[target]:>10.4f} mmol/gDCW/h')

BIOMASS_Ec_iML1515_core_75p37M            flux =     0.8770 mmol/gDCW/h
EX_etoh_e                                 flux =    20.0000 mmol/gDCW/h
EX_trp__L_e                               flux =     4.4649 mmol/gDCW/h


### Mode C — family of envelopes parameterized by w_kin

For each `w_kin`, find `v_max(w_kin)` via biomass-max with soft kinetic loss, then sweep `[0, v_max(w_kin)]` maxing product. The result: a fan of envelope curves all collinear with the Mode-B Pareto front (faint dashed backdrop), each truncating at its weight-specific `v_max(w_kin)`. A vertical drop from each truncation point to `(v_max(w_kin), 0)` marks the biomass-max corner.

> **Methods note (procedural).** The vertical drop is *not* part of the LP's natural Pareto front. It marks the biomass-max LP solution at `v = v_max(w_kin)` — the corner where efficiency tie-break gives `product = 0`. The same LP at fixed `v = v_max(w_kin)` with product-max objective rebalances to admit positive product flux (the diamond marker at the truncation). The drop visualizes "if soft Mode C voluntarily stops at v_max, here's where that lands on the x-axis."

In [20]:
# Mode C — family of envelope curves parameterized by w_kin.
# For each (product, w_kin): find v_max(w_kin), sweep [0, v_max(w_kin)], max product.
# Plot: each curve traces the Mode-B Pareto front up to v_max(w_kin). A vertical
# drop from the curve's right endpoint to (v_max(w_kin), 0) marks the biomass-max
# corner of soft Mode C — procedural, not part of the LP's natural Pareto front.

W_GRID = [0.0, 1e-4, 3e-4, 5e-4, 7e-4, 1e-3, 3e-3]
viridis = ['#440154', '#443983', '#31688e', '#21918c', '#35b779', '#90d743', '#fde725']
color_for = {w: viridis[i % len(viridis)] for i, w in enumerate(W_GRID)}

# Per-weight v_max via biomass-max with soft kinetic
v_max_per_w = {0.0: v_max_modeB}
for w in W_GRID:
    if w == 0:
        continue
    r = biomass_max_solve(
        model_e, biomass_idx, biomass_scale, maintenance,
        mode='C', w_kin=w, kinetic_targets=kinetic_targets_arr,
        glucose_bound_counts_s=glucose_bound_counts_s,
        glucose_exch_idx=glucose_exch_idx,
    )
    v_max_per_w[w] = r['v_biomass_user'] if r['status'] == 'optimal' else None

# Per-(product, weight) envelope sweep
envelopes_modeC = {short: {} for _, short in PRODUCTS}
for exch_id, short in PRODUCTS:
    secrete_idx = exchange_metabolites.index(exch_id)
    for w in W_GRID:
        v_max_w = v_max_per_w.get(w)
        if v_max_w is None or v_max_w <= 0:
            envelopes_modeC[short][w] = (np.array([0.0]), [None])
            continue
        v_grid_w = np.linspace(0, v_max_w, n_steps)
        products = []
        for v in v_grid_w:
            r = biomass_max_solve(
                model_e, biomass_idx, biomass_scale, maintenance,
                mode='C', w_kin=w, kinetic_targets=kinetic_targets_arr,
                glucose_bound_counts_s=glucose_bound_counts_s,
                glucose_exch_idx=glucose_exch_idx,
                secrete_exchange=secrete_idx,
                fix_v_biomass_user=float(v),
            )
            products.append(to_mmol(r['product_counts_s']) if r['status'] == 'optimal' else None)
        envelopes_modeC[short][w] = (v_grid_w, products)

# Plot: side-by-side family per product, with backdrop + vertical drop
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[s for _, s in PRODUCTS],
                    horizontal_spacing=0.10)
for col_i, (exch_id, short) in enumerate(PRODUCTS, start=1):
    # Mode B backdrop (faint dashed)
    v_b, p_b = envelopes_modeB[short]
    fig.add_trace(
        Scatter(x=v_b * 0.945, y=p_b, mode='lines',
                line=dict(color='lightgray', width=1.5, dash='dash'),
                name='Mode B backdrop', showlegend=(col_i == 1),
                legendgroup='backdrop', hoverinfo='skip'),
        row=1, col=col_i,
    )
    for w in W_GRID:
        v_arr, p_arr = envelopes_modeC[short][w]
        if all(p is None for p in p_arr):
            continue
        # Curve points + appended (v_max, 0) for the procedural drop
        x_plot = list(np.array(v_arr) * 0.945) + [v_arr[-1] * 0.945]
        y_plot = list(p_arr) + [0.0]
        marker_sizes  = [4] * (len(v_arr) - 1) + [13, 11]
        marker_symbols = ['circle'] * (len(v_arr) - 1) + ['diamond', 'square']
        label = 'Mode B (w=0)' if w == 0 else f'w_kin = {w:.0e}'
        fig.add_trace(
            Scatter(
                x=x_plot, y=y_plot, mode='lines+markers',
                name=label, legendgroup=label, showlegend=(col_i == 1),
                line=dict(color=color_for[w], width=2.0),
                marker=dict(color=color_for[w], size=marker_sizes,
                            symbol=marker_symbols,
                            line=dict(color='black', width=0.5)),
            ),
            row=1, col=col_i,
        )
    fig.update_xaxes(title='μ (1/h)', row=1, col=col_i)
    fig.update_yaxes(title=f'{short} secretion (mmol/gDCW/h)', row=1, col=col_i)

fig.update_layout(
    title=('Mode C — production envelopes by soft kinetic weight (GUR=10)'
           '<br><sub>Diamond = product-max LP at v=v_max(w); '
           'square = biomass-max corner; vertical drop is procedural.</sub>'),
    template='plotly_white', height=580, width=1250,
    legend=dict(orientation='v', x=1.02, y=1),
)
fig.show()

/Users/chris/projects/SMS/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning:

invalid value encountered in reduce



### Screen — kinetic sensitivity across native secretions

For 24 native secretion exchanges (organic acids, amino acids, alcohols), maximize product at fixed `v_biomass = 0.5` (mid-envelope) across `w_kin ∈ {0, 1e-3, 1e-1}`. Products with sensitivity to `w_kin` are kinetically-squeezed; insensitive products have non-kinetic slack on their secretion path.

In [21]:
# Screen — kinetic sensitivity of native secretion exchanges.
# At v_biomass = 0.5 (mid-envelope), max product flux across w_kin ∈ {0, 1e-3, 1e-1}.
SCREEN_CANDIDATES = [
    'D-LACTATE[p] exchange rev', 'SUC[p] exchange rev', 'ACET[p] exchange rev',
    'FORMATE[p] exchange rev', 'GLYCOLLATE[c] exchange rev', 'GLY[p] exchange rev',
    'L-ALPHA-ALANINE[p] exchange rev', 'GLT[p] exchange rev', 'L-ASPARTATE[p] exchange rev',
    'LYS[p] exchange rev', 'ARG[p] exchange rev', 'HIS[p] exchange rev',
    'PHE[p] exchange rev', 'TYR[p] exchange rev', 'TRP[p] exchange rev',
    'CYS[p] exchange rev', 'ETOH[p] exchange rev', 'GLN[p] exchange rev',
    'SER[p] exchange rev', 'THR[p] exchange rev', 'ILE[p] exchange rev',
    'LEU[p] exchange rev', 'VAL[p] exchange rev', 'MET[p] exchange rev',
]

screen_rows = []
for prod in SCREEN_CANDIDATES:
    if prod not in exchange_metabolites:
        continue
    secrete_idx = exchange_metabolites.index(prod)
    short = prod.replace(' exchange rev', '').replace('[p]', '').replace('[c]', '')
    row = {'product': short}
    p_at_w = []
    for w in [0.0, 1e-3, 1e-1]:
        r = biomass_max_solve(
            model_e, biomass_idx, biomass_scale, maintenance,
            mode='C' if w > 0 else 'B', w_kin=w,
            kinetic_targets=kinetic_targets_arr,
            glucose_bound_counts_s=glucose_bound_counts_s,
            glucose_exch_idx=glucose_exch_idx,
            secrete_exchange=secrete_idx,
            fix_v_biomass_user=0.5,
        )
        p_at_w.append(to_mmol(r['product_counts_s']) if r['status'] == 'optimal' else None)
    row['P @ w=0'] = p_at_w[0]
    row['P @ w=1e-3'] = p_at_w[1]
    row['P @ w=1e-1'] = p_at_w[2]
    if p_at_w[0] is not None and p_at_w[2] is not None and p_at_w[0] > 0.001:
        row['% drop'] = (p_at_w[0] - p_at_w[2]) / p_at_w[0] * 100
    else:
        row['% drop'] = None
    screen_rows.append(row)

screen_df = pd.DataFrame(screen_rows).set_index('product')
print(screen_df.to_string(float_format=lambda x: f'{x:>8.3f}' if pd.notna(x) else '       -'))
print()
n_squeezed = int((screen_df['% drop'].abs() > 5).sum())
print(f'Kinetically squeezed products (>5% drop at w_kin=1e-1): '
      f'{n_squeezed} of {len(screen_df)}')

/Users/chris/projects/SMS/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning:

invalid value encountered in reduce



                 P @ w=0  P @ w=1e-3  P @ w=1e-1   % drop
product                                                  
D-LACTATE         -0.000      -0.000      -0.000      NaN
SUC                7.858       7.858       7.858    0.000
ACET              15.661      15.661      15.661    0.000
FORMATE           50.640      50.640      50.640    0.000
GLYCOLLATE        15.313      15.313      15.313    0.000
GLY               15.425      15.425      15.425    0.000
L-ALPHA-ALANINE    9.301       9.301       9.301    0.000
GLT                5.843       5.843       5.843    0.000
L-ASPARTATE        8.440       8.440       8.440    0.000
LYS                3.646       3.646       3.646    0.000
ARG                3.963       3.963       3.963    0.000
HIS                3.940       3.940       3.940    0.000
PHE                2.500       2.500       2.500    0.000
TYR                2.614       2.614       2.614    0.000
TRP                2.053       2.053       2.053    0.000
CYS           

### State pickle for CLI drivers

Dump `model_e` and the supporting metadata to `out/biomass_reaction/state.pkl`. 
The drivers under `_drivers/` consume this pickle for parameter sweeps without 
re-running the upstream cells. See `_drivers/README.md` for the contract.

In [ ]:
from pathlib import Path

OUT_DIR = Path('out/biomass_reaction')
OUT_DIR.mkdir(parents=True, exist_ok=True)

state = {
    'model_e': model_e,
    'biomass_idx_e': biomass_idx,
    'biomass_scale_e': float(biomass_scale),
    'kinetic_targets_arr': kinetic_targets_arr,
    'maintenance': float(maintenance),
    'metabolites': metabolites,
    'exchange_metabolites': exchange_metabolites,
    'glucose_exch_idx': glucose_exch_idx,
    'glucose_bound_counts_s_default': float(glucose_bound_counts_s),
    'c2m_empirical': float(c2m_empirical),
    'rho_dry': float(rho_dry),
    # Mode A multi-objective default weights — retained for drivers that
    # surface an "include secretion penalty" knob.
    'objective_weights': {'secretion': 1e-3, 'kinetics': 1e-7, 'kinetics_in_range': 1e-2},
}

state_path = OUT_DIR / 'state.pkl'
with open(state_path, 'wb') as f:
    dill.dump(state, f)
print(f'pickled {state_path}  ({state_path.stat().st_size / 1e6:.1f} MB)')


## Takeaways

- **The biomass column works.** Standalone-FBA with the homeostatic-derived biomass reaction reproduces interpretable growth rates and production envelopes; coefficients agree with iML1515 within expected tolerance on the 60-met intersection (energy-currency residual is gross-vs-net accounting, not a modeling discrepancy).
- **Mode B is the canonical WCM-FBA envelope.** Linear μ in GUR; ETOH and TRP envelopes show clean carbon-balance tradeoffs.
- **Soft kinetic regularization shrinks the biomass-max corner with a sharp phase transition** between `w_kin = 5e-4` and `1e-3` at GUR=10, but does **not** bend envelope shape under the current WCM kinetic constraint set.
- **`v_max(w_kin)` is a soft optimum, not a hard upper bound.** It's where the LP voluntarily stops in biomass-max under `min(-v_biomass + w·kin_loss)`. The same LP at fixed `v = v_max(w)` with product-max objective rebalances to admit positive product flux — which is why the right endpoint of each Mode C envelope sits *on* the Mode-B line rather than at the x-axis. Methodologically important and explains the procedural vertical drop in the family plot.
- **Envelope shape invariance is structural.** It follows from the WCM kinetic constraints sitting on biomass-supporting reactions, not product secretion paths. Once a kinetic product pathway is added, envelope shape **should** bend with `w_kin`. This is the predicted next deliverable: soft Mode C will start doing real work on envelope shape exactly when the constraints we care about land.
- **Solver notes.** HiGHS canonical for vEcoli-scale LPs (CLARABEL/SCS produce silent `optimal_inaccurate`; GLOP hangs). A small `efficiency_weight=1e-9` (pFBA-style L1 on |v|) breaks futile-cycle degeneracy and stabilizes phase-transition sweeps.